# 2. Regression models and model diagnostics

This notebook is step 2 of 5. It needs the derived data from notebook 1.

| Section | What it does | Manuscript |
|---|---|---|
| 1 | Prevalence ratios (PR) and odds ratios (OR) on both scales | Results 3.3, Table 2 |
| 2 | Checks of the adjusted model | Tables SA5–SA7, Fig. SA1 |

**Why prevalence ratios?** GDM is common in this sample, and odds ratios then overstate the association. PRs come from survey-weighted Poisson regression with a log link.

**Models**

| Model | Covariates |
|---|---|
| Crude | One covariate at a time |
| Minimal | Wealth or residence, with age group and trimester (total association) |
| Adjusted | Age group, residence, wealth tertile, trimester and BMI |

## Setup

In [1]:
# Find the project folder, wherever the notebook was opened from
root <- normalizePath(getwd(), winslash = "/")
while (!file.exists(file.path(root, "gdm-project.Rproj"))) {
  if (dirname(root) == root) {
    stop("Open this notebook inside the gdm-project folder.")
  }
  root <- dirname(root)
}
setwd(root)

# Use the package versions locked in renv.lock
if (!nzchar(Sys.getenv("RENV_PROJECT"))) {
  Sys.setenv(RENV_CONFIG_NAMESPACES_CHECK = "FALSE")
  source("renv/activate.R")
}
options(width = 120)

# Settings (config/config.yml) and analysis functions (src/)
source(here::here("src", "common", "setup.R"))

# Data saved by notebook 1
women <- readRDS(data_file("women"))
sample <- readRDS(data_file("analytic_sample"))
des <- svy_domain(sample, women)

## 1. Prevalence ratios and odds ratios

Functions: `src/common/regression.R`.

In [2]:
x <- cfg$covariates # covariates of the adjusted model
confounders <- c("age_group", "trimester") # covariates of the minimal model

# Minimal model: each exposure with the confounders only
minimal_effects <- function(des, outcome) {
  do.call(
    rbind,
    lapply(c("wealth", "residence"), function(v) {
      e <- effect_table(
        fit_pr(des, c(v, confounders), outcome),
        "PR",
        "minimal"
      )
      e[startsWith(e$term, v), ]
    })
  )
}

For each glucose scale, this loop fits:

- the crude, minimal and adjusted models (PR and OR)
- the specification of the earlier national study, with gestational weeks instead of trimester
- the model with hypertension added
- Firth's penalised logistic regression, a check for sparse-data bias

The adjusted models are saved in `output/bdhs2022/models/` for the diagnostics below.

In [3]:
effects <- list()
anchor <- list()
htn <- list()
firth <- list()
for (s in names(scales)) {
  o <- scales[[s]]
  model_pr <- fit_pr(des, x, o)
  model_or <- fit_or(des, x, o)

  # Crude, minimal and adjusted estimates
  effects[[s]] <- cbind(
    scale = s,
    rbind(
      crude_effects(des, x, "PR", o),
      minimal_effects(des, o),
      effect_table(model_pr, "PR", "adjusted"),
      crude_effects(des, x, "OR", o),
      effect_table(model_or, "OR", "adjusted")
    )
  )

  # Specification of Mazumder et al. (2022): weeks instead of trimester
  anchor[[s]] <- cbind(
    scale = s,
    rbind(
      effect_table(fit_pr(des, cfg$covariates_anchor, o), "PR", "adjusted"),
      effect_table(fit_or(des, cfg$covariates_anchor, o), "OR", "adjusted")
    )
  )

  # Adding hypertension
  x_htn <- c(x, "hypertension")
  htn[[s]] <- cbind(
    scale = s,
    rbind(
      effect_table(fit_pr(des, x_htn, o), "PR", "adjusted + hypertension"),
      effect_table(fit_or(des, x_htn, o), "OR", "adjusted + hypertension")
    )
  )

  # Firth's penalised regression
  firth[[s]] <- cbind(
    scale = s,
    rbind(
      cbind(model = "main", firth_check(sample, x, o)),
      cbind(model = "+ hypertension", firth_check(sample, x_htn, o))
    )
  )

  saveRDS(model_pr, file.path(dir_models, paste0("model_pr_", s, ".rds")))
  saveRDS(model_or, file.path(dir_models, paste0("model_or_", s, ".rds")))
}
effects <- do.call(rbind, effects)

# The released-scale model sample must equal the last row of the participant flow
stopifnot(
  unique(effects$n[
    effects$scale == "released" & effects$model == "adjusted"
  ]) ==
    tail(read_table("sample_flow")$n_remaining, 1)
)

save_table(effects, "effects_main")
save_table(do.call(rbind, anchor), "effects_anchor")
save_table(do.call(rbind, htn), "effects_hypertension")
save_table(do.call(rbind, firth), "firth")

### Key results

Adjusted PRs for the main terms. The association with wealth or urban residence depends on the glucose scale, while prevalence is lower later in pregnancy on both.

In [4]:
key <- effects[
  effects$model == "adjusted" &
    effects$measure == "PR" &
    effects$term %in% c("wealthHighest", "residenceUrban", "trimester3rd"),
  c("scale", "term", "estimate", "ci_lower", "ci_upper")
]
key[3:5] <- round(key[3:5], 2)
key

                                  scale           term estimate ci_lower ci_upper
released.residenceUrban2       released residenceUrban     1.14     0.94     1.38
released.wealthHighest2        released  wealthHighest     1.38     1.02     1.86
released.trimester3rd1         released   trimester3rd     0.66     0.53     0.83
unconverted.residenceUrban2 unconverted residenceUrban     1.95     1.34     2.86
unconverted.wealthHighest2  unconverted  wealthHighest     1.17     0.70     1.95
unconverted.trimester3rd1   unconverted   trimester3rd     0.51     0.32     0.82


## 2. Model diagnostics

The checks use the logistic version of the adjusted model.

| Check | Question it answers |
|---|---|
| Events per parameter | Is the sample large enough for the number of parameters? |
| GVIF and survey VIF | Are the covariates collinear? |
| Archer–Lemeshow test | Does the model fit? |
| Bootstrap validation | How well does it discriminate and calibrate, after optimism correction? |
| Linearity tests | Are BMI and gestational weeks linear on the logit scale? |
| Influence (DFBETAS, Cook's distance) | Do a few women drive the results? |

In [5]:
# Run a diagnostic on both scales and stack the results
by_scale <- function(f) {
  do.call(
    rbind,
    lapply(names(scales), function(s) cbind(scale = s, f(scales[[s]], s)))
  )
}

# Load a logistic model saved above
saved_or_model <- function(s) {
  readRDS(file.path(dir_models, paste0("model_or_", s, ".rds")))
}

epv <- by_scale(function(o, s) events_per_parameter(saved_or_model(s)))
save_table(epv, "epv")
print(epv)

        scale   n events non_events parameters      epv
1    released 264    167         97          7 13.85714
2 unconverted 267     91        176          7 13.00000


### Collinearity

This concerns the covariates only, which are the same on both scales.

In [6]:
vif <- collinearity(saved_or_model("released"), sample, cfg)
print(vif)
save_table(vif$gvif, "vif_generalised")
save_table(vif$survey_vif, "vif_survey")

$gvif
               term     gvif df gvif_adjusted
age_group age_group 1.072351  1      1.035544
residence residence 1.070395  1      1.034599
wealth       wealth 1.097276  2      1.023479
trimester trimester 1.076387  2      1.018573
bmi             bmi 1.106418  1      1.051864

$survey_vif
          column survey_vif ordinary_vif       zeta     varrho  r_squared
1   age_group25+  1.0229669     1.030367 0.16936002   5.862172 0.02947247
2 residenceUrban  9.5545530     1.065122 0.01700126 527.630445 0.06114080
3   wealthMiddle  0.8550325     1.308219 0.12234542   5.342130 0.23560194
4  wealthHighest  1.4151018     1.428457 0.21348357   4.640406 0.29994409
5   trimester2nd  1.1797282     1.708064 0.13256943   5.209961 0.41454175
6   trimester3rd  1.2125840     1.770171 0.15397865   4.448732 0.43508267
7            bmi  1.0749024     1.084102 0.22297818   4.446685 0.07757789



### Fit, validation, linearity and influence

Bootstrap validation refits the model in 1,000 bootstrap samples on each scale, which makes it the slowest step here (a few seconds per scale). The seed is fixed in `config/config.yml`, so results are reproducible.

In [7]:
save_table(
  by_scale(function(o, s) archer_lemeshow(sample, cfg, outcome = o)),
  "archer_lemeshow"
)

validation <- by_scale(function(o, s) bootstrap_validation(sample, cfg, o))
save_table(validation, "bootstrap_validation")
print(validation)

save_table(
  by_scale(function(o, s) calibration_groups(sample, cfg, outcome = o)),
  "calibration_groups"
)
save_table(
  by_scale(function(o, s) linearity_tests(sample, women, cfg, o)),
  "linearity_tests"
)

influence <- by_scale(function(o, s) influence_checks(sample, women, cfg, o))
save_table(influence, "influence_refits")
print(influence)

        scale   measure      apparent      optimism     corrected    B
1    released       auc  6.918946e-01  3.881069e-02  6.530839e-01 1000
2    released intercept -3.886577e-10 -3.513832e-05  3.513793e-05 1000
3    released     slope  1.000000e+00  2.023545e-01  7.976455e-01 1000
4    released     brier  2.091497e-01 -1.348517e-02  2.226348e-01 1000
5 unconverted       auc  6.822552e-01  4.066169e-02  6.415936e-01 1000
6 unconverted intercept -5.028948e-13  2.499722e-03 -2.499722e-03 1000
7 unconverted     slope  1.000000e+00  1.981300e-01  8.018700e-01 1000
8 unconverted     brier  2.021396e-01 -1.284983e-02  2.149894e-01 1000
                       scale                                 scenario n_dropped           term  estimate  ci_lower
residenceUrban      released                                all women         0 residenceUrban 1.1380636 0.9402902
wealthHighest       released                                all women         0  wealthHighest 1.3789490 1.0240419
trimester3rd    

**Next:** `03_inequality`.